# 🏢 Employee Attrition Prediction
**AIML Summer Internship 2026 — Capstone Project**

**Problem Statement:** Predict whether an employee will leave the company (Attrition: Yes/No)

**Dataset:** IBM HR Analytics Employee Attrition Dataset (from Kaggle)

---
### Project Phases:
1. Data Loading & EDA
2. Data Preprocessing & Feature Engineering
3. Model Building (Logistic Regression, Random Forest, XGBoost)
4. Model Evaluation
5. Save Best Model
6. Deploy with Streamlit

## Phase 1: Install & Import Libraries

In [ ]:
# Install required libraries (only needed in Colab)
!pip install xgboost --quiet
!pip install streamlit --quiet

In [ ]:
# Import all libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Evaluation
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve
)

# Save model
import pickle

print('✅ All libraries imported successfully!')

## Phase 2: Load Dataset

**Download dataset from Kaggle:**
👉 https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset

Upload the CSV file to Colab using the Files panel on the left, OR run the Kaggle API cell below.

In [ ]:
# OPTION A: Upload manually
# Click the folder icon on the left > Upload > select WA_Fn-UseC_-HR-Employee-Attrition.csv

# OPTION B: Load directly from URL (no signup needed)
url = 'https://raw.githubusercontent.com/dsrscientist/dataset1/master/WA_Fn-UseC_-HR-Employee-Attrition.csv'
df = pd.read_csv(url)

print(f'Dataset shape: {df.shape}')
print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')
df.head()

## Phase 3: Exploratory Data Analysis (EDA)

In [ ]:
# Basic info
print('=== Dataset Info ===')
print(df.info())
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Basic Stats ===')
df.describe()

In [ ]:
# Target variable distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
df['Attrition'].value_counts().plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'], edgecolor='black')
axes[0].set_title('Attrition Count', fontsize=13)
axes[0].set_xticklabels(['No (Stayed)', 'Yes (Left)'], rotation=0)
axes[0].set_ylabel('Count')

# Pie chart
df['Attrition'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%',
                                     colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[1].set_title('Attrition Distribution', fontsize=13)
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('attrition_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Class imbalance ratio:', df['Attrition'].value_counts(normalize=True).to_dict())

In [ ]:
# Attrition by key features
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

features = ['Department', 'JobRole', 'MaritalStatus', 'OverTime', 'Gender', 'BusinessTravel']

for i, feature in enumerate(features):
    ax = axes[i//3][i%3]
    attrition_rate = df.groupby(feature)['Attrition'].apply(
        lambda x: (x == 'Yes').mean() * 100
    ).reset_index()
    attrition_rate.columns = [feature, 'Attrition Rate (%)']
    
    sns.barplot(data=attrition_rate, x=feature, y='Attrition Rate (%)',
                palette='RdYlGn_r', ax=ax)
    ax.set_title(f'Attrition Rate by {feature}', fontsize=11)
    ax.tick_params(axis='x', rotation=30)
    ax.set_ylabel('Attrition Rate (%)')

plt.tight_layout()
plt.savefig('attrition_by_features.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap (numeric features only)
plt.figure(figsize=(14, 10))
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap='coolwarm',
            linewidths=0.5, square=True, fmt='.2f')
plt.title('Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## Phase 4: Data Preprocessing

In [ ]:
# Make a copy
df_clean = df.copy()

# Drop columns with no variation (useless for ML)
cols_to_drop = ['EmployeeCount', 'Over18', 'StandardHours', 'EmployeeNumber']
df_clean.drop(columns=cols_to_drop, inplace=True)
print(f'Dropped: {cols_to_drop}')

# Encode target variable
df_clean['Attrition'] = (df_clean['Attrition'] == 'Yes').astype(int)
print(f'Target encoded: Yes=1, No=0')

print(f'\nShape after cleaning: {df_clean.shape}')

In [ ]:
# Encode categorical features
le = LabelEncoder()
categorical_cols = df_clean.select_dtypes(include=['object']).columns.tolist()
print(f'Categorical columns: {categorical_cols}')

# Store encoders for Streamlit app later
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col])
    label_encoders[col] = le
    print(f'  {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}')

print('\n✅ All categorical columns encoded!')

In [ ]:
# Split features and target
X = df_clean.drop('Attrition', axis=1)
y = df_clean['Attrition']

print(f'Features (X): {X.shape}')
print(f'Target (y): {y.shape}')
print(f'Attrition rate: {y.mean()*100:.1f}%')

# Save feature names for Streamlit
feature_names = X.columns.tolist()
print(f'\nTotal features: {len(feature_names)}')

In [ ]:
# Train-Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape}')
print(f'Test set: {X_test.shape}')

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('✅ Data split and scaled!')

## Phase 5: Model Building (3 Models)

In [ ]:
# ── Model 1: Logistic Regression ──
print('Training Logistic Regression...')
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)
lr_proba = lr_model.predict_proba(X_test_scaled)[:, 1]
print('✅ Logistic Regression trained!')

In [ ]:
# ── Model 2: Random Forest ──
print('Training Random Forest...')
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)  # No scaling needed for tree-based
rf_pred = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)[:, 1]
print('✅ Random Forest trained!')

In [ ]:
# ── Model 3: XGBoost ──
print('Training XGBoost...')
# Handle class imbalance with scale_pos_weight
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_weight = neg_count / pos_count

xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    scale_pos_weight=scale_weight,
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False
)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]
print('✅ XGBoost trained!')

## Phase 6: Model Evaluation

In [ ]:
# Compare all models
def evaluate_model(name, y_true, y_pred, y_proba):
    return {
        'Model': name,
        'Accuracy':  round(accuracy_score(y_true, y_pred) * 100, 2),
        'Precision': round(precision_score(y_true, y_pred) * 100, 2),
        'Recall':    round(recall_score(y_true, y_pred) * 100, 2),
        'F1 Score':  round(f1_score(y_true, y_pred) * 100, 2),
        'ROC-AUC':   round(roc_auc_score(y_true, y_proba) * 100, 2)
    }

results = pd.DataFrame([
    evaluate_model('Logistic Regression', y_test, lr_pred, lr_proba),
    evaluate_model('Random Forest',       y_test, rf_pred, rf_proba),
    evaluate_model('XGBoost',             y_test, xgb_pred, xgb_proba)
])

results = results.set_index('Model')
print('=== Model Comparison ===')
print(results.to_string())

# Highlight best model
best_model_name = results['F1 Score'].idxmax()
print(f'\n🏆 Best Model (by F1 Score): {best_model_name}')

In [ ]:
# Visualization: Model comparison bar chart
results.plot(kind='bar', figsize=(12, 5), colormap='Set2', edgecolor='black')
plt.title('Model Comparison — All Metrics (%)', fontsize=13)
plt.ylabel('Score (%)')
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ROC Curves
fig, ax = plt.subplots(figsize=(8, 6))

for name, proba in [('Logistic Regression', lr_proba), ('Random Forest', rf_proba), ('XGBoost', xgb_proba)]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — All Models', fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion Matrix for best model
model_preds = {
    'Logistic Regression': lr_pred,
    'Random Forest': rf_pred,
    'XGBoost': xgb_pred
}
best_pred = model_preds[best_model_name]

cm = confusion_matrix(y_test, best_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Stayed', 'Left'],
            yticklabels=['Stayed', 'Left'])
plt.title(f'Confusion Matrix — {best_model_name}', fontsize=13)
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nClassification Report — {best_model_name}:')
print(classification_report(y_test, best_pred, target_names=['Stayed', 'Left']))

In [ ]:
# Feature Importance (Random Forest)
importances = rf_model.feature_importances_
feat_imp = pd.Series(importances, index=X.columns).sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 6))
feat_imp.plot(kind='barh', color='steelblue', edgecolor='black')
plt.title('Top 15 Feature Importances — Random Forest', fontsize=13)
plt.xlabel('Importance Score')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## Phase 7: Save Best Model

In [ ]:
# Determine which model object to save
best_models = {
    'Logistic Regression': (lr_model, True),   # (model, needs_scaling)
    'Random Forest': (rf_model, False),
    'XGBoost': (xgb_model, False)
}

best_model_obj, needs_scaling = best_models[best_model_name]

# Save model + scaler + encoders + feature names together
model_package = {
    'model': best_model_obj,
    'scaler': scaler if needs_scaling else None,
    'needs_scaling': needs_scaling,
    'label_encoders': label_encoders,
    'feature_names': feature_names,
    'model_name': best_model_name
}

with open('model.pkl', 'wb') as f:
    pickle.dump(model_package, f)

print(f'✅ Model saved as model.pkl')
print(f'   Model type: {best_model_name}')
print(f'   Features: {len(feature_names)}')

In [ ]:
# Download model.pkl to your local machine
from google.colab import files
files.download('model.pkl')
print('⬇️ Download started! Place model.pkl inside Model/ folder on GitHub.')

## ✅ Summary

| Phase | Status |
|-------|--------|
| Data Loading | ✅ Done |
| EDA | ✅ Done |
| Preprocessing | ✅ Done |
| Model Training (3 models) | ✅ Done |
| Evaluation | ✅ Done |
| Model Saved | ✅ Done |

**Next Step:** Use `app.py` to deploy on Streamlit Cloud.

---
*Dataset: IBM HR Analytics | Models: LR, RF, XGBoost*